In [12]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


### Change the labels from string to integers and stratified split the dataset

In [13]:
## replace Positive as 1 and Negative as 0
df = pd.read_csv('dataset_120.csv')
mapping_dict = {'Positive': 1, 'Negative': 0}
df['label'] = df['label'].map(mapping_dict)
df.to_csv('updated_dataset_120.csv', index=False)

## split train test dataset in stratified way. Train and test dataset have equal proportion of positives and negatives
labels = df.iloc[:, 1]
indices = range(len(df))
train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.3, 
    stratify=labels, 
    random_state=42
)

print("Train dataset value counts: ",labels[train_idx].value_counts())
print()
print("Test dataset value counts: ",labels[test_idx].value_counts())

Train dataset value counts:  label
0    42
1    42
Name: count, dtype: int64

Test dataset value counts:  label
0    18
1    18
Name: count, dtype: int64


### Create custom Dataloaders

In [ ]:

class StrepDataset(Dataset):
    def __init__(self, csv, img_dir, transform = None):
        self.data = pd.read_csv(csv)
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_name).convert("RGB")

        if self.transform:
            image = self.transform(image)
        
        features = torch.tensor(self.data.iloc[idx, 2:9])

        label = torch.tensor(self.data.iloc[idx, 1])

        return image, features, label

transforms_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transforms_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = StrepDataset(csv='updated_dataset_120.csv', img_dir='data', transform=transforms_train)
test_dataset = StrepDataset(csv='updated_dataset_120.csv', img_dir='data', transform=transforms_test)

## Created Subsets for the 
train_subset = Subset(train_dataset, train_idx)
test_subset = Subset(test_dataset, test_idx)

train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=16, shuffle=True)

### Multimodal network with Resnet18 as the CNN(feature extraction) and MLP for the 7 symptoms

In [15]:
class StrepClassifier(nn.Module):
    def __init__(self, num_features, num_classes=1):
        super(StrepClassifier, self).__init__()
        
        # 1. Image Branch: Pre-trained ResNet18
        # remove the final fully connected layer to get visual embeddings
        resnet = models.resnet18(weights='DEFAULT')

        for param in resnet.parameters():
            param.requires_grad = False
        
        for param in resnet.layer4.parameters():
            param.requires_grad = True
        
        self.image_branch = nn.Sequential(*list(resnet.children())[:-1])

        # 2. Feature Branch: Simple MLP for numerical data
        self.feature_branch = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        # # 3. Fusion Layer: Combines image embeddings (512) + feature embeddings (32)
        self.classifier = nn.Sequential(
            nn.Linear(512 + 32, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes), # Output for 2 classes
            nn.Sigmoid()
        )

    def forward(self, images, features):
        # Process image
        img_out = self.image_branch(images)
        img_out = torch.flatten(img_out, 1) # Shape: [batch, 512]
        
        # # Process features
        feat_out = self.feature_branch(features) # Shape: [batch, 32]
        
        # # Concatenate branches
        combined = torch.cat((img_out, feat_out), dim=1) # Shape: [batch, 544]

        # # Final classification
        return self.classifier(combined)

### Train the model

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StrepClassifier(num_features = 7, num_classes = 1).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001, weight_decay=1e-2)

epochs = 25

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for images, features, labels in train_loader:
        images, features, labels = images.to(device), features.to(device), labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(images, features.float()).squeeze() ## [batch_size, 1] -> [batch_size]
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_loss += loss.item()

    print("Epoch:", epoch)
    print("Train loss:", train_loss)

    with torch.no_grad(): # Disable gradient calculation to save memory/speed
        val_loss = 0.0
        correct = 0.0
        total = 0.0
        for images, features, labels in test_loader:
            images, features, labels = images.to(device), features.to(device).float(), labels.to(device).float()
            
            outputs = model(images, features).squeeze()
            val_loss += criterion(outputs, labels).item()
            
            # Convert probabilities to binary predictions (0 or 1)
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    # Print progress
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(test_loader)
    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val Acc: {accuracy:.2f}%")
        

C:\Users\avk1028\AppData\Local\Temp\ipykernel_26924\1880705268.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  features = torch.tensor(self.data.iloc[idx, 2:9])


Epoch: 0
Train loss: 4.233886659145355
Epoch [1/25] Train Loss: 0.7056 | Val Loss: 0.7411 | Val Acc: 33.33%
Epoch: 1
Train loss: 4.123909175395966
Epoch [2/25] Train Loss: 0.6873 | Val Loss: 0.6808 | Val Acc: 52.78%
Epoch: 2
Train loss: 4.0196656584739685
Epoch [3/25] Train Loss: 0.6699 | Val Loss: 0.6584 | Val Acc: 63.89%
Epoch: 3
Train loss: 3.7289756536483765
Epoch [4/25] Train Loss: 0.6215 | Val Loss: 0.6637 | Val Acc: 55.56%
Epoch: 4
Train loss: 3.381876766681671
Epoch [5/25] Train Loss: 0.5636 | Val Loss: 0.6785 | Val Acc: 58.33%
Epoch: 5
Train loss: 3.420457422733307
Epoch [6/25] Train Loss: 0.5701 | Val Loss: 0.7658 | Val Acc: 58.33%
Epoch: 6
Train loss: 3.2372580766677856
Epoch [7/25] Train Loss: 0.5395 | Val Loss: 0.6964 | Val Acc: 52.78%
Epoch: 7
Train loss: 2.966839462518692
Epoch [8/25] Train Loss: 0.4945 | Val Loss: 0.7604 | Val Acc: 55.56%
Epoch: 8
Train loss: 2.7387507557868958
Epoch [9/25] Train Loss: 0.4565 | Val Loss: 0.6517 | Val Acc: 55.56%
Epoch: 9
Train loss: 2.2